# Enzyme Sequence Homology Clustering (MMseqs2)

## Objective
Group all sequences in the curated dataset (`data/processed/curated/curated_pet_dataset.csv`) into sequence-similarity clusters using **MMseqs2** at 70%, 50%, and 30% sequence identity, inspect the cluster size distributions, and export `data/processed/clustering/curated_pet_clusters.csv`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

## 1. Load Curated Dataset & Export Sequences to FASTA

In [ ]:
import pandas as pd
from src.data import export_sequences_to_fasta

curated_csv_path = project_root / "data" / "processed" / "curated" / "curated_pet_dataset.csv"
clustering_dir = project_root / "data" / "processed" / "clustering"
if not curated_csv_path.exists():
    raise FileNotFoundError(
        f"{curated_csv_path} not found. Run notebooks/01_data_curation.ipynb first."
    )

dataset = pd.read_csv(curated_csv_path)
fasta_path = export_sequences_to_fasta(dataset, clustering_dir / "curated_pet_sequences.fasta")
print(f"Loaded {len(dataset)} curated sequences and exported FASTA to: {fasta_path}")

## 2. Run MMseqs2 Clustering & Inspect Distributions

In [ ]:
import tempfile
from src.clustering import cluster_dataset_by_identity
from src.visualizations import plot_cluster_size_distributions

identity_thresholds = [0.70, 0.50, 0.30]
mmseqs_work_dir = Path(tempfile.gettempdir()) / "mmseqs2_runs"
clustered_dataset, cluster_summary = cluster_dataset_by_identity(
    dataset=dataset,
    input_fasta_path=fasta_path,
    output_directory=mmseqs_work_dir,
    identity_thresholds=identity_thresholds,
)

display(cluster_summary)
plot_cluster_size_distributions(clustered_dataset, identity_thresholds=identity_thresholds)

## 3. Save Clustered Dataset

In [ ]:
clustered_csv_path = clustering_dir / "curated_pet_clusters.csv"
columns_to_save = [
    "protein_id",
    "protein_name",
    "sequence_length",
    "uniprot_accession",
    "organism_name",
    "pet_degradation_label",
    "evidence_source",
    "cluster_id_70",
    "cluster_id_50",
    "cluster_id_30",
    "sequence",
]

clustered_dataset[columns_to_save].to_csv(clustered_csv_path, index=False)
print(f"Saved clustered dataset to: {clustered_csv_path}")
clustered_dataset[columns_to_save[:-1]].head()